# Prerequisites

## Load Bronze data to dataframe

In [0]:
%sql
use catalog ecommerce;
create schema if not exists silver;


In [0]:
orders_df = spark.read.table("bronze.orders")
orders_df.limit(5).display()

In [0]:
orders_df.printSchema()

# Transformations

## Remove Duplicate

In [0]:
from pyspark.sql.functions import *
orders_df.groupBy('order_id').count().filter(col("count")>1).display()

In [0]:
orders_df = orders_df.dropDuplicates()

## Validate customer IDs (no Nulls and >= 0)

In [0]:
from pyspark.sql.functions import col
orders_df = orders_df.filter(col("customer_id").isNotNull()).filter(col("customer_id")>=0)

In [0]:
orders_df.limit(5).display() 

## Standardize status.

In [0]:
valid_statuses = [
    "PLACED",
    "CONFIRMED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

orders_df = orders_df.withColumn("order_status", upper(trim(col("order_status")))).filter(col("order_status").isin(valid_statuses))
    

In [0]:
display(orders_df.count())
orders_df.limit(5).display()

## Validate order dates

In [0]:
orders_df = orders_df.filter(
    (col("order_date").isNotNull())
    & (col("order_date") <= current_timestamp())
)

In [0]:
display(orders_df.count())
orders_df.limit(5).display()

## Reject or quarantine negative order amounts

In [0]:
orders_df = orders_df.filter(col("total_amount") >= 0)
display(orders_df.count())
orders_df.limit(5).display()

## Identify cancelled orders , Identify returned orders , Identify completed orders

In [0]:
cancelled_df = orders_df.filter(upper(col("order_status")) == "CANCELLED")
display(cancelled_df.count())
cancelled_df.limit(5).display()

In [0]:
returned_df = orders_df.filter(upper(col("order_status")) == "RETURNED")
display(returned_df.count())
returned_df.limit(5).display()

In [0]:
completed_df = orders_df.filter(upper(col("order_status")) == "DELIVERED")
display(completed_df.count())
completed_df.limit(5).display()

# Create Silver table

## Write to silver schema

In [0]:
%sql
--drop table if exists ecommerce.silver.orders;

In [0]:
orders_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.orders")

## Read silver.orders table

In [0]:
spark.read.table("ecommerce.silver.orders").display()